# 25-Gaussian mixture — Anisotropic

## Imports

In [ ]:
import math
import sys
import numpy as np
import torch
from matplotlib import pyplot as plt

sys.path.insert(0, "..")
from src.schr_bridge import Schr_Bridge
torch.set_num_threads(1)

## Configuration

In [ ]:
SEED = 0xBADBEEF
B = 0.7260482230229087
MU = -0.0709625619633558
FIGURE_NAME = "anisotropic_ou_50.png"

## Published Table 1 row
The five rows below are the recorded evaluations of the published fit, in sampling order. This cell recalculates the printed mean and standard deviation directly from those measurements. The remaining cells independently refit the model with the fixed parameters above and evaluate that new fit.

In [ ]:
def print_table_row(rows):
    for label, key, scale, digits in [
        ("Sliced Wasserstein", "sw", 1, 3),
        ("10^5 x squared MMD", "mmd2", 100000, 2),
        ("Energy distance", "energy", 1, 4),
    ]:
        values = scale * np.array([row[key] for row in rows])
        print(f"{label}: {values.mean():.{digits}f} +/- {values.std(ddof=0):.{digits}f}")

published_evaluations = [
    {'sw': 0.1156638041138649, 'mmd2': 1.5532510185378712e-05, 'energy': 0.0036602020263671875},
    {'sw': 0.09982810169458389, 'mmd2': -7.09731248549339e-06, 'energy': 0.002776622772216797},
    {'sw': 0.15503059327602386, 'mmd2': 0.00026138335777714516, 'energy': 0.006869792938232422},
    {'sw': 0.10335256159305573, 'mmd2': 2.420392060664689e-05, 'energy': 0.0030837059020996094},
    {'sw': 0.11567006260156631, 'mmd2': 7.953840503160947e-05, 'energy': 0.00379180908203125},
]
print_table_row(published_evaluations)

## Distributions

In [ ]:
rng = np.random.RandomState(SEED)
means = np.array([(i, j) for i in range(-6, 7, 3) for j in range(-6, 7, 3)])
cov = np.tile(0.1 * np.eye(2), (25, 1, 1))
for i in range(25):
    a = rng.uniform(0,np.pi)
    rot = np.array([[np.cos(a),-np.sin(a)],[np.sin(a),np.cos(a)]])
    cov[i] = rot @ np.diag(rng.uniform(.05,.5,2)) @ rot.T
centers = torch.tensor(means,dtype=torch.float32)
chol = torch.tensor(np.linalg.cholesky(cov),dtype=torch.float32)

In [ ]:
def target(n):
    i = torch.randint(25,(n,))
    return centers[i]+(chol[i]@torch.randn(n,2,1)).squeeze(-1)

def source(n):
    parts = []
    while n:
        x = torch.randn(int(n/.9)+16,2)
        x = x[x.square().sum(-1)<=-2*math.log(.1)][:n]
        parts.append(x)
        n -= len(x)
    return torch.cat(parts)

## Training

In [ ]:
def train(b, m, seed):
    torch.manual_seed(seed)
    model = Schr_Bridge(2,50,.1,True,128,b=b,m=m,T=10.)
    model.init_r_by_samples(target(50))
    optimizer = torch.optim.Adam(model.parameters(),lr=.002)
    decay = math.exp(-10*b)
    invK = math.exp(5*math.e**2*math.sqrt(2)*math.log1p(-decay**2)
                    -2*math.e**2*math.sqrt(2)*math.asin(decay))
    tau = model.ou_sigma(model.b,10.)
    for step in range(5000):
        y = target(128)
        q = y.square().sum(-1)/(.2*tau)
        logp = model.get_log_potential(y)
        penalty = (logp+q-q.max()-3).relu().mean()
        loss = -logp.mean()+invK*model.get_log_C_tilde(10.)+5*penalty
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if not torch.isfinite(loss):
        raise ValueError("nonfinite training loss")
    return model.eval()

In [ ]:
model = train(B, MU, SEED + 2)

## Evaluation and Figure 1
Evaluate the same fit five times, using 10,000 endpoints and 10,000 fresh targets each time. Sliced Wasserstein uses 100 directions. For the Gaussian kernel, pool the three squared-distance matrices to determine the bandwidth; omit within-sample diagonals in squared MMD. Report the mean and standard deviation (divisor five). The first evaluation supplies the plotted samples.

In [ ]:
def sw(x,y):
    directions = torch.randn(2,100)
    directions /= directions.norm(dim=0)
    return ((x@directions).sort(0).values-(y@directions).sort(0).values).abs().mean().item()

def metrics(x,y):
    # Freeze the same 10,000 evaluation points for every term of each metric.
    dxx,dyy,dxy = (torch.cdist(a,b) for a,b in [(x,x),(y,y),(x,y)])
    ed = (2*dxy.mean()-dxx.mean()-dyy.mean()).item()
    result = dict(sw=sw(x,y),energy=ed)
    for d in (dxx,dyy,dxy):
        d.square_()
    gamma = .5/torch.cat([d.ravel() for d in (dxx,dyy,dxy)]).median().item()
    for d in (dxx,dyy,dxy):
        d.mul_(-gamma).exp_()
    n=len(y)
    dxx.fill_diagonal_(0)
    dyy.fill_diagonal_(0)
    result.update(gamma=gamma,mmd2=((dxx.sum(dtype=torch.float64)+dyy.sum(dtype=torch.float64))/(n*(n-1))-2*dxy.mean(dtype=torch.float64)).item())
    return result

In [ ]:
rows = []
with torch.no_grad():
    for i in range(5):
        torch.manual_seed(SEED + 100 + i)
        x, y = model(source(10000)), target(10000)
        rows.append(metrics(x, y))
        if i == 0:
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.scatter(*y.T, s=5, c="green", alpha=.3, label="Target")
            ax.scatter(*x.T, s=5, c="tab:orange", alpha=.3, label="Generated")
            ax.scatter(*means.T, s=12, c="red", label="Target centers")
            ax.set_aspect("equal")
            ax.legend()
            fig.tight_layout()
            fig.savefig(FIGURE_NAME, dpi=180)
            plt.show()


print_table_row(rows)